## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[0]))

import scripts.prepare as prepare
import scripts.services as services
import scripts.roi as roi_services
import scripts.bss_pipeline as bss_pl
import scripts.visualization as vis

from scipy.interpolate import make_interp_spline

%load_ext autoreload
%autoreload 2

### fixed variables

In [ ]:
N_PC = 10 # discovered by analysis
NFRAMES_OUT = 800
SCALE_OUT = 0.2
FPS_OUT = 240
N_PEAKS = 5
ROIS = roi_services.load_rois(str(Path.cwd().parents[0] / "rois/rois.json"))
PATH_OUT = str(Path.cwd().parents[0] / "outputs")
CAMP = "camp_3" # choose output directory for pre-processing

### Campaigns 3

#### load paths

In [ ]:
plant_names_1_m = ['../videos/camp_3/20260525/20260525_1/m/VID_20260525_073740210.mp4',
                   '../videos/camp_3/20260526/20260526_1/m/VID_20260526_073704629.mp4',
                   '../videos/camp_3/20260527/20260527_1/m/VID_20260527_073900146.mp4',
                   '../videos/camp_3/20260528/20260528_1/m/VID_20260528_060712313.mp4',
                   '../videos/camp_3/20260529/20260529_1/m/VID_20260529_083217557.mp4',
                   '../videos/camp_3/20260530/20260530_1/m/VID_20260530_074908974.mp4',
                   '../videos/camp_3/20260531/20260531_1/m/VID_20260531_074058464.mp4',
                   '../videos/camp_3/20260601/20260601_1/m/VID_20260601_075037657.mp4',
                   '../videos/camp_3/20260602/20260602_1/m/VID_20260602_082954275.mp4',
                   '../videos/camp_3/20260603/20260603_1/m/VID_20260603_091531958.mp4',
                   '../videos/camp_3/20260604/20260604_1/m/VID_20260604_083531697.mp4',
                   '../videos/camp_3/20260605/20260605_1/m/VID_20260605_085321999.mp4',
                   '../videos/camp_3/20260606/20260606_1/m/VID_20260606_073857092.mp4',
                   '../videos/camp_3/20260607/20260607_1/m/VID_20260607_081304960.mp4'                  
                   ]

plant_names_1_n = ['../videos/camp_3/20260525/20260525_1/n/VID_20260525_185103702.mp4',                 
                   '../videos/camp_3/20260526/20260526_1/n/VID_20260526_202725561.mp4',                 
                   '../videos/camp_3/20260527/20260527_1/n/VID_20260527_202243346.mp4',                 
                   '../videos/camp_3/20260528/20260528_1/n/VID_20260528_194053898.mp4',                 
                   '../videos/camp_3/20260529/20260529_1/n/VID_20260529_200346418.mp4',                 
                   '../videos/camp_3/20260530/20260530_1/n/VID_20260530_212358245.mp4',                 
                   '../videos/camp_3/20260531/20260531_1/n/VID_20260531_205308881.mp4',                 
                   '../videos/camp_3/20260601/20260601_1/n/VID_20260602_002556321.mp4',                 
                   '../videos/camp_3/20260602/20260602_1/n/VID_20260602_215851776.mp4',                 
                   '../videos/camp_3/20260603/20260603_1/n/VID_20260603_202255613.mp4',                 
                   '../videos/camp_3/20260604/20260604_1/n/VID_20260604_204613557.mp4',                 
                   '../videos/camp_3/20260605/20260605_1/n/VID_20260605_204739528.mp4',                 
                   '../videos/camp_3/20260606/20260606_1/n/VID_20260606_195750977.mp4',                 
                   '../videos/camp_3/20260607/20260607_1/n/VID_20260607_183651231.mp4'                 
                   ]

plant_names_2_m = ['../videos/camp_3/20260525/20260525_2/m/VID_20260525_073831392.mp4',
                   '../videos/camp_3/20260526/20260526_2/m/VID_20260526_073755265.mp4',
                   '../videos/camp_3/20260527/20260527_2/m/VID_20260527_074035158.mp4',
                   '../videos/camp_3/20260528/20260528_2/m/VID_20260528_060627096.mp4',
                   '../videos/camp_3/20260529/20260529_2/m/VID_20260529_083140178.mp4',
                   '../videos/camp_3/20260530/20260530_2/m/VID_20260530_074702260.mp4',
                   '../videos/camp_3/20260531/20260531_2/m/VID_20260531_074018077.mp4',
                   '../videos/camp_3/20260601/20260601_2/m/VID_20260601_074946932.mp4',
                   '../videos/camp_3/20260602/20260602_2/m/VID_20260602_082906225.mp4',
                   '../videos/camp_3/20260603/20260603_2/m/VID_20260603_091456724.mp4',
                   '../videos/camp_3/20260604/20260604_2/m/VID_20260604_083609177.mp4',
                   '../videos/camp_3/20260605/20260605_2/m/VID_20260605_085242545.mp4',
                   '../videos/camp_3/20260606/20260606_2/m/VID_20260606_073814392.mp4',
                   '../videos/camp_3/20260607/20260607_2/m/VID_20260607_081227742.mp4'
                   ]

plant_names_2_n = ['../videos/camp_3/20260525/20260525_2/n/VID_20260525_185326185.mp4',                 
                   '../videos/camp_3/20260526/20260526_2/n/VID_20260526_202842902.mp4',                 
                   '../videos/camp_3/20260527/20260527_2/n/VID_20260527_202347776.mp4',                 
                   '../videos/camp_3/20260528/20260528_2/n/VID_20260528_194140546.mp4',                 
                   '../videos/camp_3/20260529/20260529_2/n/VID_20260529_200501170.mp4',                 
                   '../videos/camp_3/20260530/20260530_2/n/VID_20260530_212439482.mp4',                 
                   '../videos/camp_3/20260531/20260531_2/n/VID_20260531_205352664.mp4',                 
                   '../videos/camp_3/20260601/20260601_2/n/VID_20260602_002637371.mp4',                 
                   '../videos/camp_3/20260602/20260602_2/n/VID_20260602_215933123.mp4',                 
                   '../videos/camp_3/20260603/20260603_2/n/VID_20260603_202219603.mp4',                 
                   '../videos/camp_3/20260604/20260604_2/n/VID_20260604_204649342.mp4',                 
                   '../videos/camp_3/20260605/20260605_2/n/VID_20260605_204823754.mp4',                 
                   '../videos/camp_3/20260606/20260606_2/n/VID_20260606_195849439.mp4',                 
                   '../videos/camp_3/20260607/20260607_2/n/VID_20260607_183615322.mp4'                
                   ]

#### pre-process the videos: do it just one time

In [ ]:
for i, video_path_in in enumerate(plant_names_1_m + plant_names_1_n + plant_names_2_m + plant_names_2_n):
    print(f'{video_path_in} processing...', end = ' ')
    video_name = video_path_in.split('/')[-1]
    prepare.pre_processing(video_path_in, PATH_OUT + f"/{CAMP}/" + video_name, NFRAMES_OUT, FPS_OUT, SCALE_OUT, ROIS[video_name])
    print(f'done.')

### Pipeline over time: each cell related to each (plant) and (day or night)

#### plant 1: day

In [ ]:
results_p1_m = {}
for video_path_in in plant_names_1_m:
    print(f'{video_path_in} processing...', end = ' ')
    
    video_name = video_path_in.split('/')[-1]
    
    unmixed, _, _ = bss_pl.run_pipeline(PATH_OUT + f"/{CAMP}/" + video_name, N_PC)
    video_out_info = services.get_video_info(PATH_OUT + f"/{CAMP}/" + video_name)
    
    fft_data = bss_pl.compute_fft_for_components(unmixed, video_out_info["fps"], range(N_PC))
    peaks_info = bss_pl.get_highest_peak_frequencies(fft_data, N_PEAKS)
    
    results_p1_m[video_path_in] = [p['highest_freq'] for p in peaks_info.values()]
    
    print(f"done.")

#### plant 1: night

In [ ]:
results_p1_n = {}
for video_path_in in plant_names_1_n:
    print(f'{video_path_in} processing...', end = ' ')
    
    video_name = video_path_in.split('/')[-1]
    
    unmixed, _, _ = bss_pl.run_pipeline(PATH_OUT + f"/{CAMP}/" + video_name, N_PC)
    video_out_info = services.get_video_info(PATH_OUT + f"/{CAMP}/" + video_name)
    
    fft_data = bss_pl.compute_fft_for_components(unmixed, video_out_info["fps"], range(N_PC))
    peaks_info = bss_pl.get_highest_peak_frequencies(fft_data, N_PEAKS)
    
    results_p1_n[video_path_in] = [p['highest_freq'] for p in peaks_info.values()]
    
    print(f"done.")

#### plant 2: day

In [ ]:
results_p2_m = {}
for video_path_in in plant_names_2_m:
    print(f'{video_path_in} processing...', end = ' ')
    
    video_name = video_path_in.split('/')[-1]
    
    unmixed, _, _ = bss_pl.run_pipeline(PATH_OUT + f"/{CAMP}/" + video_name, N_PC)
    video_out_info = services.get_video_info(PATH_OUT + f"/{CAMP}/" + video_name)
    
    fft_data = bss_pl.compute_fft_for_components(unmixed, video_out_info["fps"], range(N_PC))
    peaks_info = bss_pl.get_highest_peak_frequencies(fft_data, N_PEAKS)
    
    results_p2_m[video_path_in] = [p['highest_freq'] for p in peaks_info.values()]
    
    print(f"done.")

#### plant 2: night

In [ ]:
results_p2_n = {}
for video_path_in in plant_names_2_n:
    print(f'{video_path_in} processing...', end = ' ')
    
    video_name = video_path_in.split('/')[-1]
    
    unmixed, _, _ = bss_pl.run_pipeline(PATH_OUT + f"/{CAMP}/" + video_name, N_PC)
    video_out_info = services.get_video_info(PATH_OUT + f"/{CAMP}/" + video_name)
    
    fft_data = bss_pl.compute_fft_for_components(unmixed, video_out_info["fps"], range(N_PC))
    peaks_info = bss_pl.get_highest_peak_frequencies(fft_data, N_PEAKS)
    
    results_p2_n[video_path_in] = [p['highest_freq'] for p in peaks_info.values()]
    
    print(f"done.")

#### average all N_PC natural freqs of each video

In [ ]:
results2_p1_m = list(map(lambda x: np.mean(x), results_p1_m.values()))
results2_p1_n = list(map(lambda x: np.mean(x), results_p1_n.values()))

results2_p2_m = list(map(lambda x: np.mean(x), results_p2_m.values()))
results2_p2_n = list(map(lambda x: np.mean(x), results_p2_n.values()))

#### plot the average of natural freqs of each video

In [ ]:
dias = np.array(range(1, 15))
y_p1_m = np.array(results2_p1_m)
y_p1_n = np.array(results2_p1_n)

y_p2_m = np.array(results2_p2_m)
y_p2_n = np.array(results2_p2_n)

# cria novos pontos mais densos (suavização)
dias_smooth = np.linspace(dias.min(), dias.max(), 300)

# interpolação spline cúbica
spline_p1_m = make_interp_spline(dias, y_p1_m, k=3)
y_smooth_p1_m = spline_p1_m(dias_smooth)

spline_p1_n = make_interp_spline(dias, y_p1_n, k=3)
y_smooth_p1_n = spline_p1_n(dias_smooth)

spline_p2_m = make_interp_spline(dias, y_p2_m, k=3)
y_smooth_p2_m = spline_p2_m(dias_smooth)

spline_p2_n = make_interp_spline(dias, y_p2_n, k=3)
y_smooth_p2_n = spline_p2_n(dias_smooth)

plt.figure(figsize=(20, 10))

# curva suave
plt.plot(dias_smooth, y_smooth_p1_m, label='CP mean p1 (manhã)')
plt.plot(dias_smooth, y_smooth_p1_n, label='CP mean p1 (noite)')

plt.plot(dias_smooth, y_smooth_p2_m, label='CP mean p2 (manhã)')
plt.plot(dias_smooth, y_smooth_p2_n, label='CP mean p2 (noite)')

# pontos originais
plt.scatter(dias, y_p1_m)
plt.scatter(dias, y_p1_n)

plt.scatter(dias, y_p2_m)
plt.scatter(dias, y_p2_n)

plt.xlabel('Dia da semana', fontsize=22)
plt.ylabel('Frequência (Hz)', fontsize=22)
plt.title('Evolução das frequências médias de maior intensidade ao longo dos dias', fontsize=22)
plt.xticks(dias)
plt.legend(loc='upper right', fancybox=True, shadow=True, fontsize=20)
plt.tick_params(axis='both', labelsize=22)
# plt.ylim(0, 10)
plt.grid(True)
plt.tight_layout()
# plt.show()
plt.savefig('../outputs/camp_3_pipeline_over_time.png', bbox_inches = 'tight')